# Evaluate the group-split models (a NEW analysis: opens the test sets again)
Evaluates arms A, B and C (seed 42) that were trained on the **group-aware DDR split**, on: the group-split DDR test set, the held-out EyePACS patients, and APTOS. Then compares them with the same seed trained on the **random** split. The EyePACS and APTOS test sets are identical in both, so those rows are a direct comparison; the DDR test sets differ.

This is **not** part of your original declared analysis, so the notebook asks you to confirm a dated declaration before it opens any test set, and it logs every opening. About 10-20 minutes on a GPU (inference and bootstrap only; nothing is trained).

**Before running**
* Accelerator **GPU**. Internet is not needed.
* Add as inputs: (1) the output of your original run (`fork-of-early-dr`: images, manifest and the original seed-42 results), (2) the output of **`train_on_group_split`** (the `v2/runs` models).

**What to send me:** the comparison table at the end.

In [ ]:
import base64, glob, json, os, shutil, subprocess, sys, zlib
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
CODE = os.path.join(WORK, "code")
EXTRA = os.environ.get("EARLYDR_TEST", "").split()     # test hook, leave empty on Kaggle
CODE_FILES = (
    "..."
    "JQ0QhbQbHSIBbPrk/n8BjlZH4g=="
)
try:
    import timm  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"])
for name, text in json.loads(zlib.decompress(base64.b64decode(CODE_FILES))).items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
print("code written to", CODE)
sys.path.insert(0, CODE)

def real_dirs(pattern):
    return [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)]

def processed_in(folder):
    return [p for p in glob.glob(os.path.join(folder, "processed_*")) if os.path.isdir(p) and not os.path.islink(p)]

trained = [os.path.dirname(f) for f in glob.glob(os.path.join(INPUT, "**", "v2", "manifest.csv"), recursive=True)]
trained = [d for d in trained if os.path.isdir(os.path.join(d, "runs"))]
original = [d for d in real_dirs("manifest.csv") if processed_in(d) and os.path.isdir(os.path.join(d, "runs"))]
if not trained:
    sys.exit("STOP: add the output of train_on_group_split (it has v2/runs) as an input.")
if not original:
    sys.exit("STOP: add the output of your original run (images, manifest and runs) as an input.")
TRAINED, ORIGINAL = trained[0], original[0]
print("group-split models:", TRAINED, "\noriginal run:", ORIGINAL)

import pandas as pd
V2 = os.path.join(WORK, "v2")
os.makedirs(V2, exist_ok=True)
for name in ["manifest.csv", "splits.csv"]:
    shutil.copy(os.path.join(TRAINED, name), os.path.join(V2, name))
shutil.copytree(os.path.join(TRAINED, "runs"), os.path.join(V2, "runs"), dirs_exist_ok=True)
processed = processed_in(ORIGINAL)[0]
link = os.path.join(V2, os.path.basename(processed))
if not os.path.exists(link):
    os.symlink(processed, link)
print(pd.read_csv(os.path.join(V2, "manifest.csv")).groupby(["source", "split"]).size().unstack(fill_value=0))

## Declare this analysis, then confirm
Read the text, then set `CONFIRM_DECLARATION = True` and rerun the cell. Nothing is evaluated until you do.

In [ ]:
import datetime
TEXT = f"""# Declaration for the group-split analysis
Date declared: {datetime.date.today()}   (written BEFORE the test sets are opened for this analysis)

This is a new analysis, added after the original declared analysis and after seeing its results.
* Models: arms A, B, C, seed 42, trained on the group-aware DDR split (EyePACS and APTOS splits unchanged).
* Test sets: DDR group-split test, held-out EyePACS patients, APTOS. Each is opened once for this analysis.
* Comparison: the same seed trained on the random DDR split (original results).
* Metrics: AUROC, PR-AUC, specificity and sensitivity at the frozen threshold (0.90 specificity target), per-grade AUROC.
* The per-grade and specificity metrics are post-hoc and will be labelled as such in the thesis.
"""
CONFIRM_DECLARATION = False
if not CONFIRM_DECLARATION:
    print(TEXT)
    sys.exit("Read the declaration above, set CONFIRM_DECLARATION = True and run this cell again.")
open(os.path.join(V2, "DECLARATION_group_split.md"), "w").write(TEXT)
print("declaration saved to", os.path.join(V2, "DECLARATION_group_split.md"))

## Evaluate (internal test sets, then APTOS)

In [ ]:
for arm in ["arm_A", "arm_B", "arm_C"]:
    for flag in ([], ["--external"]):
        subprocess.run([sys.executable, "-m", "early_dr.run_experiment", f"configs/arms/{arm}.yaml", "--set", "task=stage1",
                        f"paths.work={V2}", *EXTRA, "--stage", "evaluate", "--seeds", "42", *flag], cwd=CODE)
print(open(os.path.join(V2, "test_opened.log")).read())

## Compare with the random-split models (seed 42)
Old = trained on the random DDR split. New = trained on the group split. EyePACS and APTOS rows use identical test images; the DDR rows use different test images (the split changed).

In [ ]:
import numpy as np
from early_dr.dup_report import early_metrics
rows = []
for arm in ["arm_A", "arm_B", "arm_C"]:
    for source in ["DDR", "EyePACS", "APTOS"]:
        res = {}
        for label, base in [("old", ORIGINAL), ("new", V2)]:
            folder = os.path.join(base, "runs", arm, "seed42")
            preds = pd.read_csv(os.path.join(folder, f"preds_{source}.csv"))
            thr = json.load(open(os.path.join(folder, "calibration.json")))["threshold_dr"]
            res[label] = early_metrics(preds, thr)
        for metric in ["specificity", "sens_early", "sens_grade1", "auroc_g1_vs_0", "auroc_g2_vs_0", "pr_auc", "auroc"]:
            rows.append({"arm": arm, "source": source, "metric": metric, "old (random split)": round(res["old"][metric], 3),
                         "new (group split)": round(res["new"][metric], 3), "change": round(res["new"][metric] - res["old"][metric], 3)})
table = pd.DataFrame(rows)
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)
table.to_csv(os.path.join(WORK, "audit", "group_split_comparison.csv"), index=False)
wide = table.pivot_table(index=["arm", "source"], columns="metric", values="change")[["specificity", "sens_early", "sens_grade1", "auroc_g1_vs_0", "auroc_g2_vs_0", "pr_auc", "auroc"]]
print("CHANGE (new minus old):\n"); print(wide.to_string())
print("\nFull table (old / new / change) saved to audit/group_split_comparison.csv")
print(table[table.metric.isin(["auroc", "specificity"])].to_string(index=False))

## What to send me
The printed change table (and the last table). Large changes on EyePACS or APTOS would mean DDR leakage had been helping; small changes (expected) mean the DDR split was not the problem.